# Dashboard del EDA · Encuesta ASP 2022 (Bogotá)

Proyecto integrador: segmentación predictiva de la vulnerabilidad en el acceso a salud de la población ASP en Bogotá

Integrantes: Stefany Pulido Valero, Daniela Sofía López Rivas y Lezly Juliana Nossa López

Este notebook rehace en Plotly las gráficas del EDA y genera el archivo `dashboard.json` que se publica en la página. Solo hay **estadística descriptiva de la encuesta** (nada de modelos). Reglas que sigue:

- Cada gráfica usa una **tabla agregada** (conteos, porcentajes, medias), nunca las filas de personas, porque la página es pública.
- No se muestran grupos con menos de 10 respuestas.
- Los porcentajes están **ponderados** con `factor_expansion` y sin las respuestas «No sabe/No responde» ni «No aplica» en el denominador.
- Un solo color; la barra del hallazgo va resaltada.

Se corre en orden, de arriba hacia abajo (Kernel → Restart & Run All). Requiere `pip install plotly`.

## Paso 0 · Preparar

In [1]:
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go

figuras = {}    # nombre -> gráfica de Plotly
notas = {}      # nombre -> una frase con lo que muestra la gráfica
secciones = {}  # nombre -> sección del dashboard

def pct_ponderado(df, condicion):
    """% ponderado con factor_expansion de las filas que cumplen la condición."""
    peso = df["factor_expansion"]
    return float(round(100 * peso[condicion].sum() / peso.sum(), 1))

## Paso 1 · Cargar los datos

Leemos la base de la encuesta. Para la afiliación dejamos fuera a quienes respondieron «No sabe/No responde» (quedan 2.991). Usamos los mismos términos en las tarjetas y las gráficas: colombianas y migrantes.

In [2]:
df = pd.read_csv("asp_2022_con_perfil.csv")

NSNR = ["No sabe/No responde"]
NA = "No aplica (no se hizo la pregunta)"
MINIMO = 10   # no se muestran grupos con menos de 10 respuestas

# mismo término en tarjetas y gráficas: colombianas / migrantes
df["origen"] = np.where(df["pais_nacimiento"] == "Colombia", "Colombianas",
                        np.where(df["pais_nacimiento"].isin(NSNR), "NS/NR", "Migrantes"))

# 1 = no tiene afiliación; "No sabe/No responde" sale del denominador
base = df[~df["afiliado_seg_social_salud"].isin(NSNR)].copy()
base["no_afiliada"] = (base["afiliado_seg_social_salud"] == "No").astype(int)

def _sin_vacios(d, col):
    return d[d[col].notna() & ~d[col].isin(NSNR + ["NS/NR", NA])]

def tabla_afiliacion(col, orden=None):
    """% ponderado sin afiliación por grupo (grupos con menos de MINIMO respuestas, fuera)."""
    d = _sin_vacios(base, col)
    filas = [(g, len(x), pct_ponderado(x, x["no_afiliada"] == 1)) for g, x in d.groupby(col) if len(x) >= MINIMO]
    t = pd.DataFrame(filas, columns=["grupo", "respuestas", "pct"])
    if orden:
        t = t.set_index("grupo").reindex([o for o in orden if o in set(t["grupo"])]).reset_index()
    return t

def tabla_distribucion(serie, datos=None):
    """% de la población (ponderado) en cada categoría, sin NS/NR ni 'No aplica' y sin grupos de menos de MINIMO."""
    d = (df if datos is None else datos).assign(grupo=serie)
    d = _sin_vacios(d, "grupo")
    total = d["factor_expansion"].sum()
    filas = [(g, len(x), float(round(100 * x["factor_expansion"].sum() / total, 1)))
             for g, x in d.groupby("grupo", observed=True) if len(x) >= MINIMO]
    return pd.DataFrame(filas, columns=["grupo", "respuestas", "pct"])

CLARO, OSCURO = "#D9CCEE", "#6D3FB2"   # un solo color; la barra del hallazgo va resaltada

def barras(t, titulo, etiqueta_valor, etiqueta_grupo=None, horizontal=False, resaltar=None, alto=420):
    """Barras de una tabla agregada (grupo, respuestas, pct). Resalta una sola barra (por defecto, la mayor)."""
    if resaltar is None:
        resaltar = t.loc[t["pct"].idxmax(), "grupo"]
    colores = [OSCURO if g == resaltar else CLARO for g in t["grupo"]]
    texto = [f"{v:.1f} %".replace(".", ",") for v in t["pct"]]
    tope = float(t["pct"].max()) * 1.18
    if horizontal:
        tr = go.Bar(x=t["pct"], y=t["grupo"], orientation="h", text=texto, textposition="outside", cliponaxis=False,
                    customdata=t["respuestas"], marker=dict(color=colores),
                    hovertemplate="%{y}<br>%{x:.1f} %<br>Respuestas: %{customdata}<extra></extra>")
    else:
        tr = go.Bar(x=t["grupo"], y=t["pct"], text=texto, textposition="outside", cliponaxis=False,
                    customdata=t["respuestas"], marker=dict(color=colores),
                    hovertemplate="%{x}<br>%{y:.1f} %<br>Respuestas: %{customdata}<extra></extra>")
    fig = go.Figure(data=[tr])
    if horizontal:   # sin título en el eje de las categorías; la primera categoría arriba
        fig.update_layout(xaxis=dict(range=[0, tope], title=dict(text=etiqueta_valor)),
                          yaxis=dict(autorange="reversed", automargin=True))
    else:
        fig.update_layout(xaxis=dict(title=dict(text=etiqueta_grupo)),
                          yaxis=dict(range=[0, tope], title=dict(text=etiqueta_valor)))
    fig.update_layout(title=dict(text=titulo), template="plotly_white", showlegend=False, height=alto,
                      margin=dict(t=60, l=70, r=90, b=70))
    return fig

def coma(x, d=1):
    return f"{x:,.{d}f}".replace(",", "#").replace(".", ",").replace("#", ".")

def mediana_ponderada(d, col):
    """Mediana de la columna ponderada con factor_expansion (igual que los porcentajes)."""
    d = d.sort_values(col)
    acumulado = d["factor_expansion"].cumsum() / d["factor_expansion"].sum()
    return int(d.loc[acumulado >= 0.5, col].iloc[0])

def guardar(nombre, fig, nota, seccion):
    figuras[nombre], notas[nombre], secciones[nombre] = fig, nota, seccion

## Paso 2 · Sección «Quiénes son»: edad, escolaridad, ingresos y horas de actividad

In [3]:
# =============== Sección "Quiénes son" ===============
rangos = pd.cut(df["edad"], bins=[17, 24, 29, 34, 39, 49, 120], labels=["18-24", "25-29", "30-34", "35-39", "40-49", "50 o más"])
t = tabla_distribucion(rangos.astype(str))
t = t.set_index("grupo").reindex(["18-24", "25-29", "30-34", "35-39", "40-49", "50 o más"]).reset_index()
guardar("distribucion_edad",
        barras(t, "Distribución de la población por rango de edad", "% de la población", "Edad (años)"),
        f"Es una población joven: la mitad tiene {mediana_ponderada(df, 'edad')} años o menos.", "Quiénes son")

grupo_edu = {"Ninguno": "Ninguno o preescolar", "Preescolar": "Ninguno o preescolar", "Básica primaria": "Básica primaria",
             "Básica secundaria": "Básica secundaria", "Media": "Media", "Técnico": "Técnico o tecnológico",
             "Tecnológico": "Técnico o tecnológico", "Universitario": "Universitario o más",
             "Especialización": "Universitario o más", "Doctorado": "Universitario o más"}
orden_edu = ["Ninguno o preescolar", "Básica primaria", "Básica secundaria", "Media", "Técnico o tecnológico", "Universitario o más"]
t = tabla_distribucion(df["nivel_educativo"].map(grupo_edu))
t = t.set_index("grupo").reindex(orden_edu).reset_index()
hasta_media = t.loc[t["grupo"].isin(orden_edu[:4]), "pct"].sum()
guardar("escolaridad", barras(t, "Nivel educativo más alto alcanzado", "% de la población", horizontal=True),
        f"El {coma(hasta_media)} % llegó como máximo a educación media.", "Quiénes son")

t = tabla_distribucion(df["ingresos_suficientes_gastos"]).sort_values("pct", ascending=False)
guardar("ingresos_alcanzan",
        barras(t, "¿Los ingresos alcanzan para cubrir los gastos?", "% de la población", None, resaltar="No"),
        f"El {coma(t.loc[t.grupo == 'No', 'pct'].iloc[0])} % dice que sus ingresos no alcanzan para los gastos.", "Quiénes son")

horas = pd.cut(df["horas_semana_asp"], bins=[0, 20, 40, 60, 100], labels=["Hasta 20", "21 a 40", "41 a 60", "Más de 60"])
t = tabla_distribucion(horas.astype(str))
t = t.set_index("grupo").reindex(["Hasta 20", "21 a 40", "41 a 60", "Más de 60"]).reset_index()
guardar("horas_semana", barras(t, "Horas a la semana dedicadas a la actividad", "% de la población", "Horas por semana"),
        f"El {coma(t.loc[t.grupo.isin(['41 a 60', 'Más de 60']), 'pct'].sum())} % dedica más de 40 horas a la semana.", "Quiénes son")

## Paso 3 · Sección «Dónde están»: localidad

In [4]:
# =============== Sección "Dónde están" ===============
t = tabla_distribucion(df["localidad"]).sort_values("pct", ascending=False)
guardar("distribucion_localidad",
        barras(t, "Localidad donde se hizo la encuesta (solo localidades con 10 o más respuestas)",
               "% de la población", horizontal=True, alto=600),
        f"{t.iloc[0]['grupo']} reúne el {coma(t.iloc[0]['pct'])} % de la población y {t.iloc[1]['grupo']} el {coma(t.iloc[1]['pct'])} %.",
        "Dónde están")

## Paso 4 · Sección «Acceso a salud»: afiliación, régimen y barrera de dinero

El régimen solo se pregunta a las personas afiliadas. La pregunta de barreras solo se le hizo a una parte de la muestra (804 personas), así que el porcentaje es sobre ellas. De las barreras mostramos solo la falta de dinero: las demás (maltrato, negación del servicio, acoso) tienen menos de 10 respuestas «Sí».

In [5]:
# =============== Sección "Acceso a salud" ===============
t = tabla_afiliacion("origen", ["Colombianas", "Migrantes"])
guardar("afiliacion_origen",
        barras(t, "Personas sin afiliación a salud según origen", "% sin afiliación", None, resaltar="Migrantes"),
        f"El {coma(t.loc[t.grupo == 'Migrantes', 'pct'].iloc[0])} % de las migrantes no tiene afiliación a salud, "
        f"frente al {coma(t.loc[t.grupo == 'Colombianas', 'pct'].iloc[0])} % de las colombianas.", "Acceso a salud")

orden_tiempo = ["Menos de 1 año", "Entre 1 y 5 años", "Entre 6 y 10 años", "Más de 10 años"]
t = tabla_afiliacion("tiempo_viviendo_bogota", orden_tiempo)
guardar("afiliacion_tiempo",
        barras(t, "Personas sin afiliación a salud según el tiempo viviendo en Bogotá", "% sin afiliación",
               "Tiempo viviendo en Bogotá", resaltar="Menos de 1 año"),
        f"Con menos de un año en Bogotá, {coma(t['pct'].iloc[0])} % no tiene afiliación; con más de 10 años, {coma(t['pct'].iloc[-1])} %.",
        "Acceso a salud")

# régimen de salud: solo personas afiliadas (en el resto la pregunta no se hizo)
afiliadas = df[df["afiliado_seg_social_salud"] == "Sí"]
t = tabla_distribucion(afiliadas["regimen_salud"], afiliadas).sort_values("pct", ascending=False)
guardar("regimen_salud",
        barras(t, "Régimen de salud de las personas afiliadas", "% de las afiliadas", horizontal=True, alto=300),
        f"Entre las personas afiliadas, el {coma(t.iloc[0]['pct'])} % está en el régimen {t.iloc[0]['grupo'].split(' (')[0].lower()}.",
        "Acceso a salud")

# barrera de acceso: solo la falta de dinero tiene 10 o más respuestas "Sí"; las demás barreras no se muestran
pregunta = df[df["barrera_falta_dinero"] != NA]
t = tabla_distribucion(pregunta["barrera_falta_dinero"], pregunta).sort_values("pct", ascending=False)
guardar("barrera_dinero",
        barras(t, "¿La falta de dinero fue una barrera para acceder a salud?", "% de quienes respondieron",
               None, resaltar="Sí"),
        f"Entre las {coma(len(pregunta[~pregunta['barrera_falta_dinero'].isin(NSNR)]), 0)} personas a las que se les hizo esta pregunta, para el {coma(t.loc[t.grupo == 'Sí', 'pct'].iloc[0])} % la falta de dinero fue una barrera para acceder a salud.",
        "Acceso a salud")

## Paso 5 · Cifras clave

In [6]:
# --- cifras clave (tarjetas de arriba del dashboard) ---
n_enc = len(df)
sin_afil = pct_ponderado(base, base["no_afiliada"] == 1)
d = df[df["origen"] != "NS/NR"]
migrantes = pct_ponderado(d, d["origen"] == "Migrantes")
kpis = [
    {"valor": f"{n_enc:,}".replace(",", "."), "etiqueta": "encuestas"},
    {"valor": f"{coma(sin_afil)} %", "etiqueta": "sin afiliación a salud"},
    {"valor": f"{coma(migrantes)} %", "etiqueta": "migrantes"},
    {"valor": f"{mediana_ponderada(df, 'edad')} años", "etiqueta": "edad mediana"},
]
kpis

[{'valor': '3.003', 'etiqueta': 'encuestas'},
 {'valor': '50,8 %', 'etiqueta': 'sin afiliación a salud'},
 {'valor': '45,8 %', 'etiqueta': 'migrantes'},
 {'valor': '28 años', 'etiqueta': 'edad mediana'}]

## Paso 6 · Exportar y revisar

Crea `dashboard.json` (cada gráfica lleva su `seccion`) y comprueba que pese menos de 1000 KB, que no tenga `id_persona` y el orden de las gráficas.

In [7]:
salida = {
    "kpis": kpis,
    "figuras": [{"nombre": n, "seccion": secciones[n], "nota": notas.get(n, ""), "figura": json.loads(f.to_json())}
                for n, f in figuras.items()],
}
with open("../../Cloud/web/dashboard.json", "w", encoding="utf-8") as f:
    json.dump(salida, f, ensure_ascii=False)

texto = open("../../Cloud/web/dashboard.json", encoding="utf-8").read()
print(round(len(texto.encode()) / 1024), "KB")   # tiene que ser menos de 1000
print("id_persona" in texto)                     # tiene que dar False
print([(x["seccion"], x["nombre"]) for x in salida["figuras"]])  # el orden en que saldrán

72 KB
False
[('Quiénes son', 'distribucion_edad'), ('Quiénes son', 'escolaridad'), ('Quiénes son', 'ingresos_alcanzan'), ('Quiénes son', 'horas_semana'), ('Dónde están', 'distribucion_localidad'), ('Acceso a salud', 'afiliacion_origen'), ('Acceso a salud', 'afiliacion_tiempo'), ('Acceso a salud', 'regimen_salud'), ('Acceso a salud', 'barrera_dinero')]
